<a href="https://colab.research.google.com/github/IvanVelezQu/Estructuras-base-de-datos/blob/main/Arboles%20y%20lista/B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==========================================
# IMPLEMENTACIÓN DEL ÁRBOL B+
# ==========================================

class HojaBPlus:
    def __init__(self, orden):
        self.orden = orden
        self.claves = []   # Guardará los IDs
        self.valores = []  # Guardará las tuplas con (ID, Nombre, Edad, Promedio)
        self.siguiente = None  # El puntero mágico que conecta las hojas

    def insertar(self, clave, valor):
        # Insertar manteniendo el orden dentro de la hoja
        if clave in self.claves:
            return  # Ignoramos duplicados

        for i, c in enumerate(self.claves):
            if clave < c:
                self.claves.insert(i, clave)
                self.valores.insert(i, valor)
                break
        else:
            self.claves.append(clave)
            self.valores.append(valor)

    def esta_llena(self):
        return len(self.claves) == self.orden

    def dividir(self):
        mitad = len(self.claves) // 2
        nueva_hoja = HojaBPlus(self.orden)

        # Repartimos los datos
        nueva_hoja.claves = self.claves[mitad:]
        nueva_hoja.valores = self.valores[mitad:]
        self.claves = self.claves[:mitad]
        self.valores = self.valores[:mitad]

        # Mantenemos la lista enlazada
        nueva_hoja.siguiente = self.siguiente
        self.siguiente = nueva_hoja

        # Devolvemos la clave que debe subir al nodo padre
        return nueva_hoja.claves[0], nueva_hoja

class NodoInternoBPlus:
    def __init__(self, orden):
        self.orden = orden
        self.claves = []
        self.hijos = []

    def insertar_hijo(self, clave, hijo_izq, hijo_der):
        for i, c in enumerate(self.claves):
            if clave < c:
                self.claves.insert(i, clave)
                self.hijos.insert(i + 1, hijo_der)
                break
        else:
            self.claves.append(clave)
            self.hijos.append(hijo_der)

    def esta_lleno(self):
        return len(self.claves) == self.orden

    def dividir(self):
        mitad = len(self.claves) // 2
        clave_subida = self.claves[mitad]

        nuevo_nodo = NodoInternoBPlus(self.orden)
        nuevo_nodo.claves = self.claves[mitad + 1:]
        nuevo_nodo.hijos = self.hijos[mitad + 1:]

        self.claves = self.claves[:mitad]
        self.hijos = self.hijos[:mitad + 1]

        return clave_subida, nuevo_nodo

class ArbolBPlus:
    def __init__(self, orden=5):
        self.orden = orden
        self.raiz = HojaBPlus(orden)

    # 1. Buscar un estudiante por ID
    def buscar(self, id_matricula):
        nodo = self.raiz
        # Bajamos por los nodos internos hasta encontrar la hoja correcta
        while isinstance(nodo, NodoInternoBPlus):
            for i, c in enumerate(nodo.claves):
                if id_matricula < c:
                    nodo = nodo.hijos[i]
                    break
            else:
                nodo = nodo.hijos[-1]

        # Buscamos en la hoja
        for i, c in enumerate(nodo.claves):
            if c == id_matricula:
                return nodo.valores[i]
        return None

    # 2. Insertar nuevos estudiantes
    def insertar(self, id_matricula, nombre, edad, promedio):
        valor = (id_matricula, nombre, edad, promedio)
        clave_promovida, nuevo_nodo = self._insertar_recursivo(self.raiz, id_matricula, valor)

        # Si la raíz se dividió, creamos una nueva raíz superior
        if clave_promovida is not None:
            nueva_raiz = NodoInternoBPlus(self.orden)
            nueva_raiz.claves.append(clave_promovida)
            nueva_raiz.hijos.extend([self.raiz, nuevo_nodo])
            self.raiz = nueva_raiz

    def _insertar_recursivo(self, nodo, clave, valor):
        if isinstance(nodo, HojaBPlus):
            nodo.insertar(clave, valor)
            if nodo.esta_llena():
                return nodo.dividir()
            return None, None
        else:
            indice_hijo = 0
            for i, c in enumerate(nodo.claves):
                if clave < c:
                    break
                indice_hijo += 1
            else:
                if len(nodo.claves) > 0:
                    indice_hijo = len(nodo.claves)

            clave_promovida, nuevo_hijo = self._insertar_recursivo(nodo.hijos[indice_hijo], clave, valor)

            if clave_promovida is not None:
                nodo.insertar_hijo(clave_promovida, nodo.hijos[indice_hijo], nuevo_hijo)
                if nodo.esta_lleno():
                    return nodo.dividir()
            return None, None

    # 3. Listar todos los estudiantes en orden por ID
    def listar_en_orden(self):
        estudiantes = NUM_ESTUDIANTES
        nodo = self.raiz

        # Bajamos hasta la primera hoja (la de más a la izquierda)
        while isinstance(nodo, NodoInternoBPlus):
            nodo = nodo.hijos[0]

        # Recorremos la lista enlazada de hojas (Súper rápido)
        while nodo is not None:
            estudiantes.extend(nodo.valores)
            nodo = nodo.siguiente

        return estudiantes

In [6]:
import random

# Cantidad total de registros
NUM_ESTUDIANTES = 10000

print(f"Generando lista de {NUM_ESTUDIANTES} estudiantes...")

# 1. Crear la lista ORDENADA
datos_ordenados = []

# Usamos IDs secuenciales (ej. del 100,000 al 109,999) para asegurar el orden
for i in range(NUM_ESTUDIANTES):
    id_matricula = 100000 + i
    nombre = f"Estudiante_{id_matricula}"
    edad = random.randint(17, 30)
    promedio = round(random.uniform(3.0, 5.0), 2)

    # Guardamos cada estudiante como una tupla
    datos_ordenados.append((id_matricula, nombre, edad, promedio))

# 2. Crear la lista ALEATORIA (Copia desordenada)
# Hacemos una copia superficial de la lista original
datos_aleatorios = datos_ordenados.copy()

# Desordenamos la copia
random.shuffle(datos_aleatorios)

print("¡Listas generadas con éxito!")
print(f"- Primer elemento de la lista ordenada: {datos_ordenados[0]}")
print(f"- Primer elemento de la lista aleatoria: {datos_aleatorios[0]}")

Generando lista de 10000 estudiantes...
¡Listas generadas con éxito!
- Primer elemento de la lista ordenada: (100000, 'Estudiante_100000', 19, 3.18)
- Primer elemento de la lista aleatoria: (106262, 'Estudiante_106262', 19, 4.0)


In [8]:
import time

# Creamos instancias de ArbolBPlus para la prueba
arbol_aleatorio = ArbolBPlus()
arbol_ordenado = ArbolBPlus()

# Insertamos los estudiantes en el árbol aleatorio
print("Insertando estudiantes en árbol aleatorio...")
for estudiante in datos_aleatorios:
    arbol_aleatorio.insertar(estudiante[0], estudiante[1], estudiante[2], estudiante[3])
print("Inserción en árbol aleatorio completada.")

# Insertamos los estudiantes en el árbol ordenado
print("Insertando estudiantes en árbol ordenado...")
for estudiante in datos_ordenados:
    arbol_ordenado.insertar(estudiante[0], estudiante[1], estudiante[2], estudiante[3])
print("Inserción en árbol ordenado completada.\n")

# Extraemos solo los IDs de nuestra lista para usarlos en la búsqueda.
# Los buscamos en orden aleatorio para que la prueba sea justa para ambos árboles.
ids_a_buscar = [estudiante[0] for estudiante in datos_aleatorios]

print("Iniciando prueba de LECTURA (Búsqueda de 10,000 estudiantes)...\n")

# ==========================================
# PRUEBA 1: Búsqueda en el Árbol Aleatorio (Balanceado)
# ==========================================
inicio_busqueda_aleatoria = time.time()

for id_buscar in ids_a_buscar:
    arbol_aleatorio.buscar(id_buscar)

fin_busqueda_aleatoria = time.time()
tiempo_busqueda_aleatoria = fin_busqueda_aleatoria - inicio_busqueda_aleatoria

print(f"Tiempo de lectura en Árbol Aleatorio: {tiempo_busqueda_aleatoria:.4f} segundos")

# ==========================================
# PRUEBA 2: Búsqueda en el Árbol Ordenado (Desbalanceado / Línea)
# ==========================================
inicio_busqueda_ordenada = time.time()

for id_buscar in ids_a_buscar:
    arbol_ordenado.buscar(id_buscar)

fin_busqueda_ordenada = time.time()
tiempo_busqueda_ordenada = fin_busqueda_ordenada - inicio_busqueda_ordenada

print(f"Tiempo de lectura en Árbol Ordenado: {tiempo_busqueda_ordenada:.4f} segundos")

# ==========================================
# RESULTADOS
# ==========================================
print("Conclusión de Lectura:")
# Evitamos división por cero por si la búsqueda aleatoria es ridículamente rápida
tiempo_aleatorio_seguro = tiempo_busqueda_aleatoria if tiempo_busqueda_aleatoria > 0 else 0.0001
diferencia_lectura = tiempo_busqueda_ordenada / tiempo_aleatorio_seguro

print(f"Buscar datos en el árbol insertado en orden fue {diferencia_lectura:.2f} veces más LENTO que buscar en el árbol aleatorio.")

Insertando estudiantes en árbol aleatorio...
Inserción en árbol aleatorio completada.
Insertando estudiantes en árbol ordenado...
Inserción en árbol ordenado completada.

Iniciando prueba de LECTURA (Búsqueda de 10,000 estudiantes)...

Tiempo de lectura en Árbol Aleatorio: 0.0286 segundos
Tiempo de lectura en Árbol Ordenado: 0.0593 segundos
Conclusión de Lectura:
Buscar datos en el árbol insertado en orden fue 2.07 veces más LENTO que buscar en el árbol aleatorio.
